# Install

In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', '-q'], returncode=0)

#  Import

In [2]:
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import io
from datetime import datetime, timezone
from azure.storage.blob import BlobServiceClient

# Config

In [3]:
APP_TOKEN = os.environ["SOCRATA_APP_TOKEN"]
CONN_STR = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")

API_URL = "https://data.cityofnewyork.us/resource/i4gi-tjb9.json"
LIMIT = 50000  # max per request

client = BlobServiceClient.from_connection_string(CONN_STR)
print("Config xong")

Config xong


# Hàm pull 1 tháng

In [4]:
def pull_month(year, month):
    start = f"{year}-{month:02d}-01T00:00:00"
    if month == 12:
        end = f"{year+1}-01-01T00:00:00"
    else:
        end = f"{year}-{month+1:02d}-01T00:00:00"
    
    all_rows = []
    offset = 0
    
    while True:
        params = {
            "$where": f"data_as_of >= '{start}' AND data_as_of < '{end}'",
            "$limit": LIMIT,
            "$offset": offset,
            "$order": "data_as_of ASC",
            "$$app_token": APP_TOKEN
        }
        resp = requests.get(API_URL, params=params, timeout=60)
        resp.raise_for_status()
        rows = resp.json()
        
        if not rows:
            break
            
        all_rows.extend(rows)
        offset += LIMIT
        print(f"  {year}-{month:02d}: pulled {len(all_rows)} rows...", end="\r")
        
        if len(rows) < LIMIT:
            break
    
    return all_rows

# Hàm upload parquet lên Azure

In [5]:
def upload_parquet(df, year, month):
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    
    table = pa.Table.from_pandas(df, preserve_index=False)
    buf = io.BytesIO()
    pq.write_table(table, buf)
    buf.seek(0)
    
    blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
    blob.upload_blob(buf, overwrite=True)
    print(f"Uploaded: {blob_path} ({len(df)} rows)")

# Pull toàn bộ 01/2024 đến nay

In [6]:
from datetime import date

now = date.today()
months = []
y, m = 2024, 1
while (y, m) <= (now.year, now.month):
    months.append((y, m))
    m += 1
    if m > 12:
        m = 1
        y += 1

print(f"Tổng số tháng: {len(months)}")

Tổng số tháng: 30


In [7]:
def is_already_uploaded(year, month):
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
    try:
        blob.get_blob_properties()
        return True
    except:
        return False

for year, month in months:
    if is_already_uploaded(year, month):
        print(f"Skip {year}-{month:02d} (đã có)")
        continue
        
    print(f"\nPulling {year}-{month:02d}...")
    rows = pull_month(year, month)
    
    if not rows:
        print(f"Không có data {year}-{month:02d}, bỏ qua")
        continue
    
    df = pd.DataFrame(rows)
    upload_parquet(df, year, month)

print("\nHoàn thành pull Bronze historical")

Skip 2024-01 (đã có)
Skip 2024-02 (đã có)
Skip 2024-03 (đã có)
Skip 2024-04 (đã có)
Skip 2024-05 (đã có)
Skip 2024-06 (đã có)
Skip 2024-07 (đã có)
Skip 2024-08 (đã có)
Skip 2024-09 (đã có)
Skip 2024-10 (đã có)
Skip 2024-11 (đã có)
Skip 2024-12 (đã có)
Skip 2025-01 (đã có)
Skip 2025-02 (đã có)
Skip 2025-03 (đã có)
Skip 2025-04 (đã có)
Skip 2025-05 (đã có)
Skip 2025-06 (đã có)
Skip 2025-07 (đã có)
Skip 2025-08 (đã có)
Skip 2025-09 (đã có)
Skip 2025-10 (đã có)
Skip 2025-11 (đã có)
Skip 2025-12 (đã có)
Skip 2026-01 (đã có)
Skip 2026-02 (đã có)
Skip 2026-03 (đã có)
Skip 2026-04 (đã có)
Skip 2026-05 (đã có)
Skip 2026-06 (đã có)

Hoàn thành pull Bronze historical


# Verify

In [8]:
# Kiểm tra
for year, month in months[:30]:
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
    props = blob.get_blob_properties()
    print(f"{blob_path}: {props.size:,} bytes")

bronze/dot_traffic_speeds/historical/year=2024/month=01/raw_2024_01.parquet: 6,566,301 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=02/raw_2024_02.parquet: 3,960,147 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=03/raw_2024_03.parquet: 7,209,525 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=04/raw_2024_04.parquet: 6,984,106 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=05/raw_2024_05.parquet: 7,359,568 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=06/raw_2024_06.parquet: 7,135,940 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=07/raw_2024_07.parquet: 7,159,674 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=08/raw_2024_08.parquet: 6,641,495 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=09/raw_2024_09.parquet: 7,143,603 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=10/raw_2024_10.parquet: 6,205,979 bytes
bronze/dot_traffic_speeds/historical/year=2024/month=11/raw_2024_11.pa

In [9]:
# Check vài tháng đại diện
check_months = [(2024,1), (2024,6), (2025,1), (2025,6), (2026,1), (2026,6)]

for year, month in check_months:
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    try:
        blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
        data = blob.download_blob().readall()
        df = pd.read_parquet(io.BytesIO(data))
        
        print(f"\n{'='*50}")
        print(f"{year}-{month:02d}: {len(df):,} rows, {len(df.columns)} cols")
        print(f"Columns: {list(df.columns)}")
        print(f"link_id unique: {df['link_id'].nunique()}")
        print(f"data_as_of range: {df['data_as_of'].min()} → {df['data_as_of'].max()}")
        print(f"speed range: {df['speed'].astype(float).min():.1f} → {df['speed'].astype(float).max():.1f}")
        print(f"Null counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
        print(f"Sample row:\n{df.iloc[0]}")
    except Exception as e:
        print(f"{year}-{month:02d}: {e}")


2024-01: 996,717 rows, 13 cols
Columns: ['id', 'speed', 'travel_time', 'status', 'data_as_of', 'link_id', 'link_points', 'encoded_poly_line', 'encoded_poly_line_lvls', 'owner', 'transcom_id', 'borough', 'link_name']
link_id unique: 122
data_as_of range: 2024-01-01T00:03:03.000 → 2024-01-29T09:07:12.000
speed range: 0.0 → 108.7
Null counts:
Series([], dtype: int64)
Sample row:
id                                                                      330
speed                                                                 39.76
travel_time                                                             126
status                                                                    0
data_as_of                                          2024-01-01T00:03:03.000
link_id                                                             4329507
link_points               40.75719,-73.99724 40.76017,-74.00382 40.76185...
encoded_poly_line                                  mkwwFvqsbMsQbh@oIlTmYp}@
encoded_poly